In [0]:
CREATE OR REPLACE TABLE data_lake_hermes_ai.prata.validation_totvs AS (
SELECT
b.CD_CLIENTE,
m.MRR_12M,
b.DS_PROD,
b.DS_LIN_REC,
b.CIDADE,
b.DS_CNAE,
CASE 
  WHEN b.DS_CNAE = 'PESSOA FISICA (SEM CNAE)' THEN 1 
  ELSE 0
END AS is_pessoa_fisica,
b.DS_SEGMENTO,
b.DS_SUBSEGMENTO,
b.FAT_FAIXA,
CASE 
    WHEN b.FAT_FAIXA = 'Sem Informações de Faturamento' THEN 'Sem informações'
    WHEN b.FAT_FAIXA IN ('Faixa 00 - Ate 4,5 M','Faixa 01 - De 4,5 M ate 7,5 M','Faixa 02 - De 7,5 M ate 15 M') THEN 'Baixa'
    WHEN b.FAT_FAIXA IN ('Faixa 03 - De 15 M ate 25 M','Faixa 04 - De 25 M ate 35 M','Faixa 05 - De 35 M ate 50 M','Faixa 06 - De 50 M ate 75 M') THEN 'Média'
    WHEN b.FAT_FAIXA IN ('Faixa 07 - De 75 M ate 150 M','Faixa 08 - De 150 M ate 300 M','Faixa 09 - De 300 M ate 500 M','Faixa 10 - De 500 M ate 850 M','Faixa 11 - Acima de 850 M') THEN 'Alta'
    ELSE 'Sem informações'
END AS categoria_faturamento,
b.MARCA_TOTVS,
b.MODAL_COMERC,
b.PAIS,
b.PERIODICIDADE,
b.SITUACAO_CONTRATO,
b.UF,
CASE
    WHEN b.UF IN ('AC','AM','AP','PA','RO','RR','TO')
        THEN 'Norte'
    WHEN b.UF IN ('AL','BA','CE','MA','PB','PE','PI','RN','SE')
        THEN 'Nordeste'
    WHEN b.UF IN ('DF','GO','MT','MS')
        THEN 'Centro-Oeste'
    WHEN b.UF IN ('ES','MG','RJ','SP')
        THEN 'Sudeste'
    WHEN b.UF IN ('PR','RS','SC')
        THEN 'Sul'
    ELSE 'Indefinido'
END AS regiao,
ROUND(COALESCE(TRY_CAST(REPLACE(b.VL_TOTAL_CONTRATO, ',', '.') AS DOUBLE), 0),2) AS VL_TOTAL_CONTRATO,
b.DT_ASSINATURA_CONTRATO,
a.CLIENTE_DESDE,
date_diff(b.DT_ASSINATURA_CONTRATO,a.CLIENTE_DESDE) as Tempo_relacionamento
FROM data_lake_hermes_ai.bronze.clientes_desde a 
LEFT JOIN data_lake_hermes_ai.bronze.dados_clientes b ON a.CLIENTE=b.CD_CLIENTE
LEFT JOIN data_lake_hermes_ai.bronze.mrr m ON m.CLIENTE=b.CD_CLIENTE
WHERE PAIS=105
)


In [0]:
SELECT 
COUNT(*),
DS_CNAE
FROM data_lake_hermes_ai.prata.validation_totvs
GROUP BY 2